In [2]:
# ============================================================
# Program Kontrol Motor ESP32 via Serial (Auto-Detect Port)
# Mengirim '1' untuk memutar motor, '0' untuk mematikan motor
# ============================================================

import serial
import serial.tools.list_ports
import time
import sys

# ============================================
# KONFIGURASI
# ============================================
BAUDRATE = 115200  # Harus sama dengan Serial.begin() di ESP32

# Kata kunci pada deskripsi port yang menandakan ESP32 / USB-UART
ESP32_KEYWORDS = [
    "cp210", "cp2102", "cp210x",       # Silicon Labs (umum di ESP32 DevKit)
    "ch340", "ch341", "ch34x",         # WCH (umum di ESP32 clone)
    "ftdi", "ft232",                   # FTDI
    "usb-serial", "usb serial",
    "usb uart", "uart bridge",
    "silicon labs",
    "espressif", "esp32",
]

# (VID, PID) yang umum untuk USB-to-UART ESP32
ESP32_VID_PID = {
    (0x10C4, 0xEA60),  # Silicon Labs CP210x
    (0x10C4, 0xEA70),  # Silicon Labs CP2105
    (0x1A86, 0x7523),  # WCH CH340
    (0x1A86, 0x55D4),  # WCH CH9102
    (0x0403, 0x6001),  # FTDI FT232
    (0x0403, 0x6015),  # FTDI FT231X
    (0x303A, 0x1001),  # Espressif USB JTAG/Serial
}


# ============================================
# Fungsi untuk melihat daftar port yang tersedia
# ============================================
def list_serial_ports():
    """Menampilkan semua port serial yang terdeteksi."""
    ports = serial.tools.list_ports.comports()
    if not ports:
        print("Tidak ada port serial yang terdeteksi.")
        return []
    print("Port serial yang tersedia:")
    for p in ports:
        vid_pid = ""
        if p.vid is not None and p.pid is not None:
            vid_pid = f" [VID:PID={p.vid:04X}:{p.pid:04X}]"
        print(f"  - {p.device} : {p.description}{vid_pid}")
    return list(ports)


# ============================================
# Fungsi auto-detect port ESP32
# ============================================
def deteksi_port_esp32(verbose=True):
    """
    Mendeteksi port serial ESP32 secara otomatis.
    Return: nama port (str) atau None jika tidak ditemukan.
    """
    ports = serial.tools.list_ports.comports()
    if not ports:
        if verbose:
            print("[!] Tidak ada port serial yang terdeteksi.")
        return None

    kandidat = []

    for p in ports:
        desc = (p.description or "").lower()
        mfg = (p.manufacturer or "").lower()
        hwid = (p.hwid or "").lower()
        teks = f"{desc} {mfg} {hwid}"

        skor = 0

        # 1) Cocokkan VID:PID
        if p.vid is not None and p.pid is not None:
            if (p.vid, p.pid) in ESP32_VID_PID:
                skor += 100

        # 2) Cocokkan kata kunci deskripsi/manufacturer
        for kw in ESP32_KEYWORDS:
            if kw in teks:
                skor += 50
                break

        # 3) Hindari port Bluetooth (sering muncul di Windows)
        if "bluetooth" in teks:
            skor -= 200

        if skor > 0:
            kandidat.append((skor, p))

    if not kandidat:
        # Fallback: kalau cuma ada 1 port non-bluetooth, pakai itu
        non_bt = [p for p in ports
                  if "bluetooth" not in (p.description or "").lower()]
        if len(non_bt) == 1:
            if verbose:
                print(f"[i] Tidak ada ciri ESP32 yang cocok, "
                      f"tapi hanya ada 1 port non-Bluetooth: {non_bt[0].device}")
            return non_bt[0].device

        if verbose:
            print("[!] Port ESP32 tidak dapat dideteksi otomatis.")
            print("    Silakan tentukan PORT secara manual.")
        return None

    # Urutkan berdasarkan skor tertinggi
    kandidat.sort(key=lambda x: x[0], reverse=True)

    if verbose:
        print("Kandidat port ESP32 (skor tertinggi = paling mungkin):")
        for skor, p in kandidat:
            print(f"  [{skor:>4}] {p.device} : {p.description}")

    return kandidat[0][1].device


# ============================================
# Membuka koneksi serial
# ============================================
def buka_koneksi(port, baudrate=115200):
    """Membuka koneksi serial ke ESP32."""
    try:
        ser = serial.Serial(port, baudrate, timeout=1)
        time.sleep(2)  # Tunggu ESP32 reset setelah koneksi dibuka
        print(f"Terhubung ke ESP32 di {port} @ {baudrate} baud")
        return ser
    except serial.SerialException as e:
        print(f"Gagal membuka port {port}: {e}")
        return None


# ============================================
# Fungsi kirim perintah ke ESP32
# ============================================
def kirim_perintah(ser, perintah: str):
    """
    Mengirim perintah ke ESP32.
    perintah: '1' untuk ON, '0' untuk OFF
    """
    if perintah not in ('0', '1'):
        raise ValueError("Perintah harus '0' atau '1'")

    ser.write(perintah.encode('utf-8'))
    ser.flush()
    time.sleep(0.1)

    # Baca balasan dari ESP32 (opsional)
    if ser.in_waiting > 0:
        balasan = ser.readline().decode('utf-8', errors='ignore').strip()
        if balasan:
            print(f"ESP32: {balasan}")


def motor_on(ser):
    """Menyalakan motor (kirim '1')."""
    print(">> Mengirim '1' : Motor ON")
    kirim_perintah(ser, '1')


def motor_off(ser):
    """Mematikan motor (kirim '0')."""
    print(">> Mengirim '0' : Motor OFF")
    kirim_perintah(ser, '0')


# ============================================
# PROGRAM UTAMA
# ============================================
if __name__ == "__main__":

    # 1) Tampilkan port yang tersedia
    print("=" * 50)
    list_serial_ports()
    print("=" * 50)

    # 2) Auto-detect port ESP32
    PORT = deteksi_port_esp32()
    if PORT is None:
        print("Program dihentikan: port ESP32 tidak ditemukan.")
        sys.exit(1)

    print(f"\n>> Port ESP32 terpilih: {PORT}")

    # 3) Buka koneksi serial
    esp32 = buka_koneksi(PORT, BAUDRATE)

    if esp32 is None:
        print("Koneksi gagal. Periksa kembali PORT dan kabel USB.")
    else:
        try:
            # 4) Contoh urutan kontrol motor
            print("\n--- Mulai urutan kontrol motor ---")

            motor_on(esp32)
            time.sleep(3)        # Motor berputar 3 detik

            motor_off(esp32)
            time.sleep(1)        # Jeda 1 detik

            motor_on(esp32)
            time.sleep(2)        # Motor berputar 2 detik

            motor_off(esp32)
            print("--- Urutan kontrol selesai ---\n")

            # 5) Mode interaktif (opsional)
            print("Mode interaktif: ketik '1'=ON, '0'=OFF, 'q'=keluar")
            while True:
                cmd = input("Perintah: ").strip()
                if cmd == 'q':
                    break
                elif cmd in ('0', '1'):
                    kirim_perintah(esp32, cmd)
                else:
                    print("Perintah tidak valid! Gunakan '1', '0', atau 'q'.")

        except KeyboardInterrupt:
            print("\nDihentikan oleh pengguna.")

        finally:
            # 6) Pastikan motor mati & tutup koneksi
            try:
                motor_off(esp32)
            except Exception:
                pass
            esp32.close()
            print("Koneksi serial ditutup.")

Port serial yang tersedia:
  - COM11 : Silicon Labs CP210x USB to UART Bridge (COM11) [VID:PID=10C4:EA60]
  - COM7 : Standard Serial over Bluetooth link (COM7)
  - COM6 : Standard Serial over Bluetooth link (COM6)
Kandidat port ESP32 (skor tertinggi = paling mungkin):
  [ 150] COM11 : Silicon Labs CP210x USB to UART Bridge (COM11)

>> Port ESP32 terpilih: COM11
Terhubung ke ESP32 di COM11 @ 115200 baud

--- Mulai urutan kontrol motor ---
>> Mengirim '1' : Motor ON
ESP32: Testing DC Motor...LED is on !!!
>> Mengirim '0' : Motor OFF
ESP32: LED is off !!!
>> Mengirim '1' : Motor ON
ESP32: LED is on !!!
>> Mengirim '0' : Motor OFF
ESP32: LED is off !!!
--- Urutan kontrol selesai ---

Mode interaktif: ketik '1'=ON, '0'=OFF, 'q'=keluar


Perintah:  1


ESP32: LED is on !!!


Perintah:  0


ESP32: LED is off !!!


Perintah:  q


>> Mengirim '0' : Motor OFF
ESP32: LED is off !!!
Koneksi serial ditutup.
